# RSICD Adapter-CLIP — Session 4 of 5 — Full fine-tune baseline

Auto-generated from the rsicd-clip-adapter repo. Source of truth: `KAGGLE_RUNBOOK.md`.

**Before running this notebook:**
1. **Data** — pick ONE of these:
   - **Option A** (recommended): Click **+ Add data** in the right panel, search `rsicd-image-caption-dataset`, click **Add**
   - **Option B** (if you don't want to attach a Dataset): Download `train.csv`, `valid.csv`, `test.csv` from the thedevastator Kaggle dataset page, then in the Files panel (right), click **Upload** and drop the 3 files. They land in `/kaggle/working/`.
2. **Previous-session Datasets** (sessions 2-5 only): click **+ Add data** → add any `rsicd-adapter-s*` or `rsicd-fullfinetune` Datasets you saved earlier
3. Settings: **Accelerator = GPU P100 or T4**, **Internet = ON**
4. Click **Save Version → Save Output** at the end of this session

Cell 2 will auto-detect which option you used and report back.

In [ ]:
# === Install + clone + env ===
!pip install open_clip_torch faiss-cpu ftfy accelerate pyyaml -q
!git clone https://github.com/Vatsal057/rsicd-clip-adapter.git
%cd rsicd-clip-adapter
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
print("Repo cloned, deps installed.")

In [ ]:
# === Sanity: GPU + dataset location ===
import torch, os
from pathlib import Path
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
print()

# The data can come from two places:
#   (A) Attached Kaggle Dataset: /kaggle/input/rsicd-image-caption-dataset/
#   (B) Direct upload to /kaggle/working/ (train.csv, valid.csv, test.csv)
candidate_paths = [
    Path("/kaggle/input/rsicd-image-caption-dataset"),
    Path('/kaggle/input/rsicd-image-caption-dataset'),
    Path('/kaggle/input/rsicd-dataset'),
    Path('/kaggle/input/rsicd'),
    Path('/kaggle/working'),         # <- direct upload
    Path('/kaggle/working/rsicd-clip-adapter'),  # if upload landed in repo dir
]
for p in Path('/kaggle/input').iterdir() if Path('/kaggle/input').exists() else []:
    if p.is_dir() and (p / 'train.csv').exists():
        candidate_paths.append(p)

data_path = None
for p in candidate_paths:
    if p.exists() and (p / 'train.csv').exists():
        data_path = p
        break

if data_path is None:
    print("ERROR: Could not find an RSICD dataset.")
    print()
    print("Option A — Attach as Kaggle Dataset (preferred):")
    print("  1. Click '+ Add data' in the right panel")
    print("  2. Search 'rsicd-image-caption-dataset' (the thedevastator version)")
    print("  3. Click 'Add'")
    print()
    print("Option B — Upload CSVs directly (faster, but you re-upload every session):")
    print("  1. Download train.csv, valid.csv, test.csv from the thedevastator")
    print("     Kaggle dataset page")
    print("  2. In the Files panel (right), click 'Upload' and drop the 3 CSVs")
    print("  3. They land in /kaggle/working/")
    print()
    print("Currently attached under /kaggle/input/:")
    try:
        attached = sorted(p.name for p in Path('/kaggle/input').iterdir())
    except FileNotFoundError:
        print("   (no /kaggle/input/ directory)")
        attached = []
    for name in attached:
        print(f"   - {name}")
    print()
    print("Currently in /kaggle/working/:")
    try:
        working = sorted(p.name for p in Path('/kaggle/working').iterdir())
    except FileNotFoundError:
        print("   (no /kaggle/working/ directory)")
        working = []
    for name in working:
        print(f"   - {name}")
    raise SystemExit(0)  # Stop here so the user can fix and re-run

os.environ['RSICD_ARCHIVE'] = str(data_path)
print(f"Using dataset: {data_path}")
csvs = sorted(p.name for p in data_path.glob('*.csv'))
print(f"  CSVs:   {csvs}")

## Required: attach the previous session's Dataset
Click **+ Add data** → search `rsicd-adapter-s3` → **Add** (we use its environment but the full FT script is independent).

In [ ]:
# === Restore checkpoint from previous session ===
import shutil, os
from pathlib import Path
src = Path("/kaggle/input/rsicd-adapter-s3/adapter_best.pt")
if not src.exists():
    print(f"WARN: {src} not found. Did you forget to '+ Add data' rsicd-adapter-s3?")
else:
    Path("results/checkpoints").mkdir(parents=True, exist_ok=True)
    shutil.copy(src, "results/checkpoints/adapter_best.pt")
    print(f"Restored: {src}")
    # Also restore training history if it was saved
    src_h = Path("/kaggle/input/rsicd-adapter-s3/training_history_adapter.json")
    if src_h.exists():
        Path("results/metrics").mkdir(parents=True, exist_ok=True)
        shutil.copy(src_h, "results/metrics/training_history_adapter.json")
        print("Restored training history.")

In [ ]:
# === Full fine-tune baseline ===
!python scripts/04_run_fullfinetune.py

In [ ]:
# === Package output for next session ===
# After this cell, click 'Save Version' (top right) with 'Save Output' enabled.
# Then go to the Output tab -> 'New Dataset' -> name it 'rsicd-fullfinetune'.
# The next session will attach this Dataset via '+ Add data'.
import shutil, os
out_dir = "/kaggle/working/rsicd-fullfinetune"
os.makedirs(out_dir, exist_ok=True)
src = "results/checkpoints/fullfinetune_best.pt"
if os.path.exists(src):
    dst = os.path.join(out_dir, os.path.basename(src)) if not os.path.isdir(src) else out_dir
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(out_dir, os.path.basename(src)), dirs_exist_ok=True)
    else:
        shutil.copy(src, dst)
    print(f"  + {src} -> {out_dir}")
src = "results/metrics/fullfinetune_results.json"
if os.path.exists(src):
    dst = os.path.join(out_dir, os.path.basename(src)) if not os.path.isdir(src) else out_dir
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(out_dir, os.path.basename(src)), dirs_exist_ok=True)
    else:
        shutil.copy(src, dst)
    print(f"  + {src} -> {out_dir}")
print(f"\nReady. Save this notebook version with 'Save Output' ON, then convert output to Dataset 'rsicd-fullfinetune'.")

## Done!
Save the version, then convert output to Dataset `rsicd-fullfinetune`.

**Heads up:** the full-fine-tune checkpoint is ~600 MB. Kaggle Datasets cap at ~20 GB so it's fine, but downloading it from the notebook output to your Mac will take a while. Don't commit it to GitHub.